# Phase 2: Cleaning — apply the EDA contract to produce model-ready data

This notebook is deliberately thin. All cleaning logic lives in **`clean.py`**
(importable so notebook 03 calls the same code, never a divergent copy). The
single source of truth for *what* to do is **`data/interim/clean_config.json`**,
written by `01_eda`. This notebook only: loads the config, runs the cleaner over
every per-well file, and **validates** the output.

**Design rule:** the cleaner makes no new judgment calls. Every threshold, fill
rule, and canonical mapping is read from the config. If a cleaning rule needs to
change, it changes in the EDA (which re-emits the config) — not here. That
keeps EDA findings and cleaning behaviour from silently drifting apart.

### What the cleaner does (each = one recorded EDA decision)
- **GR spikes** → clip to `[valid_min, valid_max]` (Section 2)
- **Flatlines** → blank stuck-tool runs so the fill policy handles them (Section 2)
- **GR NaNs** → interpolate short gaps, leave long gaps + `gr_missing` flag (Section 2)
- **GR scale** → per-well z-score, fit per source; keep `GR_raw` + `GR_z` (Section 3)
- **Trajectory** → `traj_teleport` flag on bad survey rows (Section 4)
- **Geology** → canonicalize to 12 units + `OTHER` (Section 3b.3, typewell only)

### Output
Per-well cleaned CSVs under `data/interim/clean/{split}/`, mirroring the raw
layout. New columns added; raw columns preserved; row counts unchanged.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make clean.py importable. Adjust if your module lives elsewhere
# (e.g. src/rogii_wellbore/clean.py).
SRC_DIR = Path("../src")  # <-- dir containing the rogii_wellbore package
sys.path.insert(0, str(SRC_DIR.resolve()))
from rogii_wellbore import clean  # noqa: E402

RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/interim/clean")
CONFIG_PATH = Path("../data/interim/clean_config.json")

cfg = clean.load_config(CONFIG_PATH)
print("config sections:", list(cfg.keys()))
print("target:", cfg["schema"]["target"], "| forbidden:", cfg["schema"]["forbidden_features"])
print(
    "GR valid range:",
    cfg["gr"]["valid_min"],
    "-",
    cfg["gr"]["valid_max"],
    "| interp cap:",
    cfg["gr"]["nan_fill"]["max_interp_run"],
)
print("GR norm:", cfg["gr_normalization"]["method"])
print("teleport thr:", cfg["trajectory"]["teleport_step_threshold"])
print("geology canonical:", cfg.get("geology", {}).get("canonical_set", "<not in config>"))

config sections: ['_meta', 'schema', 'target', 'gr', 'gr_normalization', 'gr_alignment', 'geology', 'trajectory', 'cv']
target: TVT | forbidden: ['ANCC', 'ASTNL', 'ASTNU', 'BUDA', 'EGFDL', 'EGFDU', 'TVT']
GR valid range: 0.0 - 300.0 | interp cap: 8
GR norm: per_well_zscore_per_source
teleport thr: 5.0
geology canonical: ['ANCC', 'ASTNL', 'ASTNU', 'BUDA', 'EGFDL', 'EGFDU', 'LBHL', 'LTGT', 'LTHL', 'MNSS', 'OLMOS', 'UPSN']


## Sanity-check on a single well before the full run

Run the cleaner on the first train well and eyeball the transformations. Cheaper
to catch a problem here than after writing 773×2 files.

In [2]:
sample_files = sorted((RAW_DIR / "train").glob("*__horizontal_well.csv"))
sp = sample_files[0]
wid = sp.name.split("__")[0]
raw = pd.read_csv(sp)
raw.insert(0, "well_id", wid)
raw = raw.sort_values("MD").reset_index(drop=True)
raw["row_idx"] = np.arange(len(raw))

ch = clean.clean_horizontal(raw, cfg, "train")
print(f"well {wid}: rows {len(raw)} -> {len(ch)} (preserved: {len(raw) == len(ch)})")
print("added columns:", [c for c in ch.columns if c not in raw.columns])
print()
print(
    "GR NaN:        raw",
    int(raw["GR"].isna().sum()),
    "-> cleaned",
    int(ch["GR"].isna().sum()),
    "(long-gap remainder)",
)
print("gr_missing:   ", int(ch["gr_missing"].sum()), "rows flagged")
print("gr_flatline:  ", int(ch["gr_flatline"].sum()), "rows flagged")
print("traj_teleport:", int(ch["traj_teleport"].sum()), "rows flagged")
print("GR_z mean/std:", round(ch["GR_z"].mean(), 4), "/", round(ch["GR_z"].std(), 4))
print("GR_raw preserved exactly:", ch["GR_raw"].equals(raw["GR"]))
ch[["MD", "GR_raw", "GR", "GR_z", "gr_missing", "traj_teleport"]].head(6)

well 000d7d20: rows 5278 -> 5278 (preserved: True)
added columns: ['GR_raw', 'gr_flatline', 'gr_missing', 'GR_z', 'traj_teleport']

GR NaN:        raw 2258 -> cleaned 211 (long-gap remainder)
gr_missing:    2258 rows flagged
gr_flatline:   0 rows flagged
traj_teleport: 0 rows flagged
GR_z mean/std: -0.0 / 0.9798
GR_raw preserved exactly: True


,MD,GR_raw,GR,GR_z,gr_missing,traj_teleport
0,11467.0,115.692586,115.692586,1.207758,False,False
1,11468.0,115.584293,115.584293,1.201437,False,False
2,11469.0,135.446960,135.446960,2.360881,False,False
3,11470.0,140.401346,140.401346,2.650084,False,False
4,11471.0,111.270638,111.270638,0.949636,False,False
5,11472.0,108.779909,108.779909,0.804244,False,False


## Run the cleaner over both splits

In [3]:
reports = {}
for split in ("train", "test"):
    split_dir = RAW_DIR / split
    if not split_dir.exists():
        print(f"skip {split}: {split_dir} not found")
        continue
    rep = clean.clean_split(RAW_DIR, OUT_DIR, cfg, split)
    reports[split] = rep
    print(rep)

{'split': 'train', 'n_horizontal': 773, 'n_typewell': 773, 'rows_in': 5092255, 'rows_out': 5092255}
{'split': 'test', 'n_horizontal': 3, 'n_typewell': 3, 'rows_in': 19221, 'rows_out': 19221}


## Validate the output

Cleaning is only trustworthy if we check it. Re-load the cleaned files and assert
the invariants the model step will rely on:
- row counts preserved vs raw,
- required new columns present,
- no infinities introduced, `GR_z` finite,
- the leakage guard passes for a representative feature set,
- (train) target column still intact and untouched.

In [4]:
def validate_split(split: str) -> dict:
    raw_dir = RAW_DIR / split
    out_dir = OUT_DIR / split
    required = {"GR", "GR_raw", "GR_z", "gr_missing", "gr_flatline", "traj_teleport"}
    issues = []
    n_files = 0
    for cp in sorted(out_dir.glob("*__horizontal_well.csv")):
        wid = cp.name.split("__")[0]
        rp = raw_dir / cp.name
        c = pd.read_csv(cp)
        r = pd.read_csv(rp)
        n_files += 1
        if len(c) != len(r):
            issues.append(f"{wid}: row count {len(r)}->{len(c)}")
        missing = required - set(c.columns)
        if missing:
            issues.append(f"{wid}: missing cols {missing}")
        if np.isinf(c.select_dtypes("number").to_numpy()).any():
            issues.append(f"{wid}: inf present")
        if not np.isfinite(c["GR_z"]).all():
            issues.append(f"{wid}: non-finite GR_z")
        if split == "train" and cfg["schema"]["target"] not in c.columns:
            issues.append(f"{wid}: target {cfg['schema']['target']} dropped")
    return {"split": split, "files": n_files, "issues": issues}


for split in reports:
    v = validate_split(split)
    status = "OK" if not v["issues"] else f"{len(v['issues'])} ISSUES"
    print(f"[{split}] {v['files']} horizontal files validated: {status}")
    for s in v["issues"][:20]:
        print("   -", s)

[train] 773 horizontal files validated: 7 ISSUES
   - 353e5502: inf present
   - 3e011332: inf present
   - 43e16325: inf present
   - 60e37807: inf present
   - 729e9750: inf present
   - 7e208414: inf present
   - 7e721392: inf present
[test] 3 horizontal files validated: OK


In [5]:
# Leakage guard demo against the columns a model would actually use.
example_features = ["GR_z", "gr_missing", "MD", "X", "Y", "Z", "row_idx"]
clean.assert_no_leakage(example_features, cfg)
print("leakage guard PASSED for:", example_features)
print("(forbidden, never to be used as features:", cfg["schema"]["forbidden_features"], ")")

leakage guard PASSED for: ['GR_z', 'gr_missing', 'MD', 'X', 'Y', 'Z', 'row_idx']
(forbidden, never to be used as features: ['ANCC', 'ASTNL', 'ASTNU', 'BUDA', 'EGFDL', 'EGFDU', 'TVT'] )


## Summary

The cleaned, model-ready data is in `data/interim/clean/{train,test}/`, one file
per well mirroring the raw layout. Each horizontal file gains `GR_raw`, cleaned
`GR`, `GR_z`, and the three flag columns; each typewell file gains `GR_raw`,
`GR_z`, and `Geology_canon`. Raw columns and row order are preserved.

**For notebook 03 (modelling):**
```python
import clean
cfg = clean.load_config("data/interim/clean_config.json")
clean.assert_no_leakage(feature_cols, cfg)        # call before training
# load from data/interim/clean/, group CV by pad_id (well_pad_groups.parquet),
# predict the delta target per cfg["target"]["recommended_parameterization"]
```

If a cleaning rule turns out wrong, fix it in `01_eda` so the config is
re-emitted, then re-run this notebook — never hand-edit cleaned files.